<a href="https://colab.research.google.com/github/zzhining/test_analysis/blob/main/credit_card_fraud_prediction_skeleton.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 신용카드 사기거래 예측 모델


---


## 프로젝트 목표
---
- 주어진 데이터의 구조를 파악하고, 탐색적 데이터 분석을 수행한다.
- 주어진 데이터를 머신러닝 모델 학습에 적합한 형태로 전처리하여 학습을 수행한다.


## 프로젝트 목차
---

1. **데이터 읽기, 데이터 탐색:** 데이터를 불러오고 Dataframe 구조를 확인

2. **데이터 시각화:** 변수 시각화를 통하여 분포 파악

3. **데이터 전처리:** 머신러닝 모델에 필요한 입력값 형식으로 데이터 처리

4. **머신러닝 모델 수행:** 회귀 모델을 사용하여 학습 수행, 평가 및 예측 수행

## 데이터 출처
---
- [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)


## 프로젝트 개요
---

### 📌 배경 (Context)
회사가 신용카드 거래 사기(fraudulent credit card transactions) 를 식별하는 것은 매우 중요합니다.
이러한 기능은 고객이 직접 구매하지 않은 항목에 대해 청구되는 일을 방지하는 데 도움이 됩니다.

### 📁 데이터 설명 (Content)
이 데이터셋은 2013년 9월, 유럽 신용카드 사용자들에 의해 발생한 거래들을 포함하고 있습니다.

데이터에는 **이틀** 동안의 거래가 담겨 있으며, 총 284,807건의 거래 중 492건이 사기 거래입니다.

이 데이터는 심하게 불균형(unbalanced) 되어 있으며, **사기 거래 클래스(positive class)는 전체의 약 0.172%**에 불과합니다.

### 📊 데이터 구성 요소
이 데이터셋은 PCA(주성분 분석) 를 통해 변환된 숫자형 입력 변수만 포함하고 있습니다.

보안상의 이유로 원본 특성과 추가적인 배경 정보는 제공되지 않습니다.

### 👉 주요 변수:
- `V1` ~ `V28`: PCA로 추출된 주성분 변수
- `Time`: 첫 번째 거래 이후부터 각 거래까지의 경과 시간 (초 단위)
- `Amount`: 거래 금액. 이 값은 거래 금액에 따른 비용 민감 학습(cost-sensitive learning) 등에 활용할 수 있음
- `Class`: 타겟 변수, 사기 거래일 경우 `1`, 그렇지 않으면 `0`

### ✅ 모델 평가 지표 권장사항
클래스 간 심각한 불균형으로 인해, 정확도(accuracy) 나 혼동 행렬(confusion matrix) 만으로는 모델 성능을 제대로 평가하기 어렵습니다.

대신 **정밀도-재현율 곡선의 면적 (AUPRC, Area Under Precision-Recall Curve)** 을 사용하여 정확도를 측정하는 것이 좋습니다.




## 1. 데이터 읽기
---


### 1.1 데이터 불러오기
---

pandas를 사용하여 데이터를 읽고 dataframe 형태로 저장합니다.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('creditcard.csv')

print(df.shape)
display(df.head())

### 1.2 데이터 탐색
---


#### 1.2.1 `Class`별 데이터의 수
---
`Class`별 데이터의 갯수를 확인합니다.
- 데이터프레임의 `value_counts()`를 사용하면 각 범주의 수를 반환합니다.
- 인자로 `normalize=True`를 설정하면 비율을 반환합니다.

In [ ]:
display(df['Class'].value_counts())  # 0: 정상, 1: 사기
display(df['Class'].value_counts(normalize=True))  # 0: 정상, 1: 사기

#### 1.2.2 데이터 컬럼 별 타입 및 결측치 확인
---
데이터 프레임의 `info()`를 호출하면 컬럼 별 데이터 타입과 결측치가 아닌 데이터의 수를 확인할 수 있습니다.

In [ ]:
df.info()

#### 1.2.3 주요 통계지표
---
데이터 프레임의 수치형 변수에 대해, 주요 통계지표를 확인합니다.

In [ ]:
df.describe()

일부 컬럼(`Amount`, `Time`)만 필터링하여 통계 지표를 확인할 수 있습니다.

In [ ]:
# 기본 전처리 및 EDA
df[['Amount', 'Time']].describe()

#### 1.2.4 Class별 주요 통계 지표 확인
---
데이터 프레임의 `groupby()`를 호출하여 범주 별 데이터를 그룹화할 수 있습니다. 그룹으로 묶인 데이터는 집계연산 결과를 설정하여 값을 확인할 수 있습니다.

In [ ]:
df.groupby(by='Class').mean()

In [ ]:
df.groupby(by='Class').mean().iloc[:, :10]
# df.groupby(by='Class').mean().iloc[:, 10:20]
# df.groupby(by='Class').mean().iloc[:, 20:]

`Class`의 값에 따라 값을 구분하여 주요 통계지표를 확인할 수 있습니다.

In [ ]:
df[['Amount', 'Time','Class']].groupby(by='Class').describe().T

In [ ]:
df.groupby(by='Class')['Amount'].describe()

## 2. 데이터 시각화
---


### 2.1 `Class` 시각화
---

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(x='Class', data=df)
plt.title("Fraud(1) vs Otherwise(0)")
plt.show()

대부분의 거래가 정상 거래(0)이며, 사기 거래(1)는 극소수임

### 2.2 각 속성의 분포 시각화
---

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

n_cols = 4
n_rows = -(-len(numeric_cols) // n_cols)

plt.figure(figsize=(5 * n_cols, 4 * n_rows))

for i, col in enumerate(numeric_cols, 1):
    plt.subplot(n_rows, n_cols, i)
    sns.histplot(df[col], bins=50, kde=True)
    plt.title(col)
    plt.xlabel('')
    plt.ylabel('')

plt.tight_layout()
plt.show()

### 2.3 `Class`에 따른 `Amount` 분포 시각화
---

In [ ]:
sns.violinplot(x='Class', y='Amount', data=df)

사기 거래는 상대적으로 적고, 금액 분포가 더 퍼져 있음

In [ ]:
# Amount가 100에서 2,200 사이인 거래만 필터링
df_temp = df[(df['Amount'] > 100) & (df['Amount'] < 2200)]

# 클래스별 데이터 분리
df_0 = df_temp[df_temp['Class'] == 0]
df_1 = df_temp[df_temp['Class'] == 1]

# 서브플롯 생성
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class 0 (정상 거래)
sns.histplot(df_0['Amount'], bins=50, kde=True, ax=axes[0], color='blue')
axes[0].set_title('Class 0 (Normal)')
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Count')

# Class 1 (사기 거래)
sns.histplot(df_1['Amount'], bins=50, kde=True, ax=axes[1], color='red')
axes[1].set_title('Class 1 (Fraud)')
axes[1].set_xlabel('Amount')
axes[1].set_ylabel('Count')

plt.suptitle("Transaction Amount Distribution by Class", fontsize=16)
plt.tight_layout()
plt.show()

### 2.4 `클래스`와의 상관관계
---

In [ ]:
# 상관계수 행렬
corr_matrix = df.corr()

# 히트맵 시각화
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix[['Class']].sort_values('Class', ascending=False),
            annot=True, cmap='coolwarm')
plt.title('Correlations with the Class')
plt.show()

### 2.5 `Time` 구간별 발생 빈도 시각화
---

In [ ]:
# 거래 시간 기반 파생 변수 생성
# 하루 24시간으로 환산 (0~86400초 기준)
df['hour'] = (df['Time'] % 86400) // 3600
df.tail()

# 상대적 시간 구간 기준 분석
df['time_block'] = pd.cut(df['hour'],
                          bins=[-1, 6, 12, 18, 24],
                          labels=['0-6h', '6-12h', '12-18h', '18-24h'])

# 시간대별 사기 비율 시각화
plt.figure(figsize=(6, 4))
sns.countplot(x='time_block', hue='Class', data=df)#, hue=)
plt.title("Relative Time Block vs Fraud Count")
plt.show()

In [ ]:
# 클래스별 데이터 분리
df_0 = df[df['Class'] == 0]
df_1 = df[df['Class'] == 1]

# 서브플롯 생성
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class 0 (정상 거래)
sns.countplot(x='time_block', data=df_0, ax=axes[0])
axes[0].set_title('Class 0 (Normal)')
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Count')

# Class 1 (사기 거래)
sns.countplot(x='time_block', data=df_1, ax=axes[1])
axes[1].set_title('Class 1 (Fraud)')
axes[1].set_xlabel('Amount')
axes[1].set_ylabel('Count')

plt.suptitle("Transaction Amount Distribution by Class", fontsize=16)
plt.tight_layout()
plt.show()

## 3. 데이터 전처리
---
- 데이터 정제: 결측치 처리, 이상치 처리
- 데이터 변환

### 3.1 데이터 정제
---


#### 3.1.1 결측치 처리
---

In [ ]:
print(f'결측치 수:', df.isna().sum().sum())

In [ ]:
df.dropna(inplace=True)

#### 3.1.2 이상치 처리
---

In [ ]:
# [TO-DO]01 . z-score 기반 이상치 처리

### 3.2 데이터 변환
---


#### 3.2.1 스케일링

In [ ]:
# [TO-DO]02 .StandardScaler


#### 3.2.2 파생변수 생성

In [ ]:
# [TO-DO]03 .거래 금액 기반 파생 변수 생성
# 거래 금액 로그 변환 (금액 분포 왜도 감소)


In [ ]:
# [TO-DO]04  금액 크기 구간화 (범주형 파생)
# bins=[-1, 10, 100, 1000, 10000, np.inf]
#labels=['0~10', '10~100', '100~1K', '1K~10K', '10K+']


In [ ]:
# [TO-DO]05. 평균 이상 거래 여부 (이진 플래그 변수)


In [ ]:
# [TO-DO]06.  고금액 & 야간 거래 여부 (리스크 플래그)


#### 3.3.3 Label Encoding

In [ ]:
# [TO-DO]07.  time_block 변수 정수 변환



In [ ]:
# 원하는 순서대로 수동 매핑
bin_order = ['0~10', '10~100', '100~1K', '1K~10K', '10K+']
amount_bin_map = {bin_label: idx for idx, bin_label in enumerate(bin_order)}

# 매핑 적용
df['amount_bin_encoded'] = df['amount_bin'].map(amount_bin_map)

# 결과 확인
print(df[['amount_bin', 'amount_bin_encoded']].drop_duplicates())

In [ ]:
df.drop(['time_block', 'amount_bin'], axis=1, inplace= True)

## 4. 머신러닝 모델 수행
---


### 4.1 RandomForestClassifier
---

#### 4.1.1 데이터 분할

In [ ]:
# [TO-DO]08.  데이터 분할


#### 4.1.2 학습

In [ ]:
# [TO-DO]09. 모델 학습 (Random Forest)



In [ ]:
# [TO-DO]10. 예측


In [ ]:
# [TO-DO]11. 중요 변수 시각화 (Feature Importance)



#### 4.1.3 평가


In [ ]:
# [TO-DO]13. 평가



In [ ]:
# [TO-DO]14. Precision-Recall Curve 시각화 (추천 지표)







plt.figure(figsize=(8, 5))
plt.plot(recall, precision, marker='.')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.grid()
plt.show()

### 4.2 Feature Engineering

#### 4.2.1 불균형 데이터 처리(SMOTE)


In [ ]:
# [TO-DO]15. 불균형 데이터 처리 – SMOTE (Synthetic Minority Over-sampling Technique)



#### 4.2.2 SMOTE 데이터로 모델 재학습

In [ ]:
# [TO-DO]16. # SMOTE 데이터로 모델 재학습



In [ ]:
# 평가
y_pred_sm = model_sm.predict(X_test)
y_proba_sm = model_sm.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_sm))
print("ROC AUC Score (SMOTE):", roc_auc_score(y_test, y_proba_sm))

---